# Double-slit interference

*optics · advanced* · [open in Tensorizer](https://jackyko1991.github.io/tensorizer/#/optics/double_slit)

The 2D scalar wave equation solved by finite differences: the Laplacian is a $3\times 3$ stencil tensor applied with conv2d. A plane wave hits a wall with two slits, and the interference fringes build up on a screen behind it.

**Vector form**

$$\partial_t^2 u = c^2\,\nabla^2 u$$

**Index (tensor) form**

$$\partial_t^2 u = c^2\,\partial_i\partial_i u,\qquad (\partial_i\partial_i u)_{ab} \approx \tfrac{1}{h^2}\textstyle\sum_{kl} S_{kl}\,u_{a+k,\,b+l}$$

| Symbol | Code | Shape | Meaning |
|---|---|---|---|
| $u_{ab}$ | `u` | (Ny, Nx) | wave field on the grid, row $a = y$, column $b = x$ |
| $S_{kl}$ | `LAPLACIAN_STENCIL` | (3, 3) | $\left[\begin{smallmatrix}0&1&0\\1&-4&1\\0&1&0\end{smallmatrix}\right]$: the 5-point Laplacian stencil |
| $\partial_i\partial_i u$ | `conv2d(u[None, None], S[None, None], padding=1) / h**2` | (Ny, Nx) | repeated $i$ ⇒ summed over $x$ and $y$: the Laplacian as a convolution |
| $u^{n+1}$ | `step(u, u_prev, sigma, c, dt, h)` | (Ny, Nx) | leapfrog: $2u^n - u^{n-1} + (c\Delta t)^2\,\nabla^2 u^n$, damped by $\sigma$ |
| $\sigma_{ab}$ | `sigma` | (Ny, Nx) | sponge damping, non-zero only in the absorbing frame |
| $c\Delta t/h \le 1/\sqrt2$ | `courant(c, dt, h)` | () | CFL stability condition of the explicit scheme in 2D |
| $\langle u^2\rangle$ | `intensity_sum / n_averaged` | (Ny,) | time-averaged intensity along the screen column |

Lengths are in grid cells ($h = 1$) and $c = 1$. Dotted lines on the screen mark where the path difference is a whole number of wavelengths, $y_m = L\tan(\arcsin(m\lambda/d))$; for small angles their spacing is $\lambda L/d$. The slits are not tiny compared with $\lambda$, so each slit's own diffraction envelope dims the outer fringes.


## Setup

On Colab, choose **Runtime → Change runtime type → GPU** to run on a GPU — the code uses it automatically.

In [ ]:
import os
import subprocess
import sys

# Shared helpers (problems/_common) come from the repository; skip the clone if we are already inside it.
if not os.path.isdir("problems"):
    if not os.path.isdir("tensorizer"):
        subprocess.run(["git", "clone", "--depth", "1", "https://github.com/jackyko1991/tensorizer.git"], check=True)
    os.chdir("tensorizer")
sys.path.insert(0, os.getcwd())
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pyyaml", "tqdm"], check=True)

import torch

print("torch", torch.__version__, "· device:", "cuda" if torch.cuda.is_available() else "cpu")


## 1. `main.py` — main loop

In [ ]:
%%writefile problems/optics/double_slit/main.py
"""Main loop: parameters → tensors → solve → plot."""

from pathlib import Path

from problems._common.runtime import cli, device

from . import equation, plot, solve, tensors


def run(params: dict, out_dir: Path) -> Path:
    dev = device()
    lam, d, dist = params["wavelength"], params["slit_sep"], params["screen_dist"]
    setup = tensors.setup(d, params["slit_width"], dist, dev)

    times, frames, intensity = solve.solve(setup, lam, params["t_max"])

    print(f"Courant number c dt/h = {equation.courant(1.0, 0.5, 1.0):.2f} (stable below {equation.CFL_LIMIT:.3f})")
    print(f"small-angle fringe spacing lambda L / d = {lam * dist / d:.1f} cells")
    fringes = solve.fringe_positions(lam, d, setup.screen_x - tensors.BARRIER_X)
    return plot.animate(times, frames, intensity, setup, fringes, out_dir / "double_slit.gif")


if __name__ == "__main__":
    cli(run, __file__)


## 2. `equation.py` — physics → tensor translation

In [ ]:
%%writefile problems/optics/double_slit/equation.py
r"""Physics → tensor translation.

The scalar wave equation in two dimensions,

    \partial_t^2 u = c^2 \nabla^2 u,     index form   \partial_t^2 u = c^2\, \partial_i \partial_i u

(repeated i ⇒ summed over i ∈ {x, y}). On a grid with spacing h the Laplacian becomes a 3×3 stencil
S_{kl} slid over the field — exactly what a 2D convolution does:

    (\partial_i\partial_i u)_{ab} \approx \frac{1}{h^2} \sum_{k,l} S_{kl}\, u_{a+k,\,b+l},
    S = [[0, 1, 0], [1, -4, 1], [0, 1, 0]]

Time uses the centred second difference (leapfrog). With a damping rate σ_{ab} (non-zero only in a
"sponge" frame round the edge, so outgoing waves are absorbed instead of reflected), the equation
\partial_t^2 u + σ \partial_t u = c^2 \nabla^2 u becomes

    (1 + σΔt/2)\, u^{n+1} = 2u^n - (1 - σΔt/2)\, u^{n-1} + (cΔt)^2 \nabla^2 u^n

This explicit scheme is stable only if the Courant number cΔt/h ≤ 1/\sqrt{2} (the CFL condition in 2D).
"""

import math

import torch
import torch.nn.functional as F

# S_{kl}: shape (3, 3). Centre weight −4, the four nearest neighbours +1.
LAPLACIAN_STENCIL = torch.tensor(
    [
        [0.0, 1.0, 0.0],
        [1.0, -4.0, 1.0],
        [0.0, 1.0, 0.0],
    ]
)

CFL_LIMIT = 1 / math.sqrt(2)  # largest stable cΔt/h for the 5-point stencil in 2D


def laplacian(u: torch.Tensor, h: float) -> torch.Tensor:
    r"""(\partial_i\partial_i u)_{ab} = \sum_{kl} S_{kl} u_{a+k,b+l} / h^2.   u: (Ny, Nx) → (Ny, Nx)

    conv2d wants (batch, channel, H, W) tensors, hence the [None, None]; padding=1 treats the
    field outside the grid as zero.
    """
    kernel = LAPLACIAN_STENCIL.to(u)[None, None]  # (1, 1, 3, 3)
    return F.conv2d(u[None, None], kernel, padding=1)[0, 0] / h**2


def courant(c: float, dt: float, h: float) -> float:
    """Courant number cΔt/h: how many cells a wave crosses in one time step."""
    return c * dt / h


def step(u: torch.Tensor, u_prev: torch.Tensor, sigma: torch.Tensor, c: float, dt: float, h: float) -> torch.Tensor:
    r"""One leapfrog step  u^{n-1}, u^n → u^{n+1}  with sponge damping σ_{ab}.   All tensors (Ny, Nx)."""
    damp = 0.5 * sigma * dt  # (Ny, Nx)
    return (2 * u - (1 - damp) * u_prev + (c * dt) ** 2 * laplacian(u, h)) / (1 + damp)


## 3. `tensors.py` — PyTorch tensors

In [ ]:
%%writefile problems/optics/double_slit/tensors.py
"""Build the concrete tensors: the grid layout, the sponge damping map and the barrier with two slits.

Everything is measured in grid cells (h = 1) and the wave speed is c = 1. Arrays are indexed
u[y, x] so they can go straight into imshow.
"""

from dataclasses import dataclass

import torch

NY, NX = 240, 260  # grid size in cells
SPONGE = 20  # width of the absorbing frame
SOURCE_X = 25  # column of the plane-wave line source
BARRIER_X = 60  # first column of the barrier
BARRIER_THICKNESS = 3


@dataclass
class Setup:
    sigma: torch.Tensor  # (Ny, Nx) damping rate σ_ab, zero in the interior
    wall: torch.Tensor  # (Ny, Nx) bool, True where the barrier is
    screen_x: int  # column where the intensity is recorded


def sponge(device: torch.device, strength: float = 0.5) -> torch.Tensor:
    """σ_{ab} grows quadratically from 0 at the inner edge of the frame to `strength` at the border."""
    y = torch.arange(NY, device=device, dtype=torch.float32)
    x = torch.arange(NX, device=device, dtype=torch.float32)
    depth_y = torch.clamp(SPONGE - torch.minimum(y, NY - 1 - y), min=0) / SPONGE  # (Ny,) 0 inside → 1 at edge
    depth_x = torch.clamp(SPONGE - torch.minimum(x, NX - 1 - x), min=0) / SPONGE  # (Nx,)
    depth = torch.maximum(depth_y[:, None], depth_x[None, :])  # (Ny, Nx)
    return strength * depth**2


def barrier(slit_sep: float, slit_width: float, device: torch.device) -> torch.Tensor:
    """Wall mask (Ny, Nx): a thick vertical wall with two openings centred at y = NY/2 ± slit_sep/2."""
    y = torch.arange(NY, device=device, dtype=torch.float32)[:, None]  # (Ny, 1)
    x = torch.arange(NX, device=device)[None, :]  # (1, Nx)
    centre = NY / 2
    in_slit = ((y - centre - slit_sep / 2).abs() < slit_width / 2) | (
        (y - centre + slit_sep / 2).abs() < slit_width / 2
    )
    in_wall_columns = (x >= BARRIER_X) & (x < BARRIER_X + BARRIER_THICKNESS)
    return in_wall_columns & ~in_slit  # (Ny, Nx) by broadcasting


def setup(slit_sep: float, slit_width: float, screen_dist: float, device: torch.device) -> Setup:
    screen_x = min(BARRIER_X + int(round(screen_dist)), NX - SPONGE - 1)
    return Setup(sponge(device), barrier(slit_sep, slit_width, device), screen_x)


## 4. `solve.py` — solver

In [ ]:
%%writefile problems/optics/double_slit/solve.py
"""Leapfrog time stepping of the wave field, with a line source driving a plane wave at the slits."""

import math

import torch

from .equation import CFL_LIMIT, courant, step
from .tensors import NX, NY, SOURCE_X, Setup


def solve(
    s: Setup, wavelength: float, t_max: float, dt: float = 0.5, save_every: int = 5
) -> tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
    """Return (times (S,), field snapshots (S, Ny, Nx), running-mean intensity on the screen (S, Ny)).

    The source adds a sin(ωt) forcing along one column (a "soft" source, so reflections pass through
    it). Wall cells are clamped to u = 0 every step. Intensity ⟨u²⟩ on the screen is averaged from
    the moment the wave front could first arrive there.
    """
    c, h = 1.0, 1.0
    if courant(c, dt, h) > CFL_LIMIT:
        raise ValueError(f"unstable: Courant number {courant(c, dt, h):.3f} > {CFL_LIMIT:.3f}")

    omega = 2 * math.pi * c / wavelength
    t_arrive = (s.screen_x - SOURCE_X) / c
    u_prev = torch.zeros(NY, NX, device=s.sigma.device)  # u^{n-1}
    u = torch.zeros_like(u_prev)  # u^n
    intensity_sum = torch.zeros(NY, device=u.device)  # Σ_n u²  on the screen column
    n_averaged = 0

    times, frames, intensity = [], [], []
    for n in range(int(round(t_max / dt))):
        t = n * dt
        u_next = step(u, u_prev, s.sigma, c, dt, h)
        ramp = min(1.0, t / (2 * wavelength))  # switch the source on smoothly over ~2 periods
        u_next[:, SOURCE_X] += dt**2 * ramp * math.sin(omega * t)
        u_next[s.wall] = 0.0  # rigid, perfectly reflecting barrier
        u_prev, u = u, u_next

        if t > t_arrive:
            intensity_sum += u[:, s.screen_x] ** 2
            n_averaged += 1
        if n % save_every == 0:
            times.append(t)
            frames.append(u.clone())
            intensity.append(intensity_sum / max(n_averaged, 1))
    return torch.tensor(times), torch.stack(frames), torch.stack(intensity)


def fringe_positions(wavelength: float, slit_sep: float, screen_dist: float, max_order: int = 3) -> list[float]:
    """Bright fringes (relative to the axis) where the path difference is mλ: y_m = L tan(arcsin(mλ/d))."""
    ys = []
    for m in range(-max_order, max_order + 1):
        sin_theta = m * wavelength / slit_sep
        if abs(sin_theta) < 1:
            ys.append(screen_dist * math.tan(math.asin(sin_theta)))
    return ys


## 5. `plot.py` — 2D/3D plot → GIF

In [ ]:
%%writefile problems/optics/double_slit/plot.py
"""Animate the wave field behind the slits next to the intensity pattern building up on the screen."""

from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch

from problems._common.plotting import frame_indices, save_gif

from .tensors import BARRIER_X, NX, NY, SPONGE, Setup


def animate(
    times: torch.Tensor, frames: torch.Tensor, intensity: torch.Tensor, s: Setup, fringes: list[float], path: Path
) -> Path:
    u = frames[:, SPONGE:-SPONGE, SPONGE:-SPONGE].cpu().numpy()  # (S, ny, nx)
    screen = intensity[:, SPONGE:-SPONGE].cpu().numpy()  # (S, ny)
    wall = s.wall[SPONGE:-SPONGE, SPONGE:-SPONGE].cpu().numpy()
    t = times.cpu().numpy()
    # coordinates relative to the centre of the slits
    x0, x1 = SPONGE - BARRIER_X, NX - SPONGE - BARRIER_X
    y0, y1 = SPONGE - NY / 2, NY - SPONGE - NY / 2
    y = np.arange(u.shape[1]) + y0
    behind = u[len(u) // 2 :, :, BARRIER_X - SPONGE + 5 :]  # transmitted waves, second half of the run
    vmax = np.abs(behind).max() * 0.7 + 1e-12  # the strong incident wave simply saturates the colour map

    fig, (ax, ax_i) = plt.subplots(1, 2, figsize=(7, 4), sharey=True, gridspec_kw={"width_ratios": [3, 1]})
    field = ax.imshow(
        u[0],
        origin="lower",
        extent=(x0, x1, y0, y1),
        cmap=plt.get_cmap("RdBu_r", 11),
        vmin=-vmax,
        vmax=vmax,
        interpolation="nearest",
    )
    ax.imshow(np.ma.masked_where(~wall, wall), origin="lower", extent=(x0, x1, y0, y1), cmap="binary", vmin=0, vmax=1)
    screen_x = s.screen_x - BARRIER_X
    ax.axvline(screen_x, ls="--", color="#111827", lw=1)
    ax.set_xlabel("x (cells)")
    ax.set_ylabel("y (cells)")
    ax.grid(False)
    title = ax.set_title("")

    for yf in [f for f in fringes if y0 < f < y1]:  # where the textbook formula puts the bright fringes
        ax_i.axhline(yf, color="#9ca3af", lw=0.8, ls=":")
    (curve,) = ax_i.plot(screen[0], y, color="#dc2626")
    ax_i.set_xlim(0, screen.max() * 1.05 + 1e-12)
    ax_i.set_xticks([])
    ax_i.set_ylim(y0, y1)
    ax_i.set_xlabel("intensity ⟨u²⟩")
    ax_i.set_title("screen")

    def update(k: int):
        field.set_data(u[k])
        curve.set_xdata(screen[k])
        title.set_text(f"wave field u(x, y), t = {t[k]:.0f}")

    fig.tight_layout()
    return save_gif(fig, update, frame_indices(len(t), 72), path, fps=12)


## Run

Change the parameters (or any file above) and run again.

In [ ]:
import importlib
from pathlib import Path

from IPython.display import Image, display

from problems._common.runtime import load_meta, resolve_params

params = {
    'wavelength': 12.0,
    'slit_sep': 48.0,
    'slit_width': 6.0,
    'screen_dist': 150.0,
    't_max': 450.0
}

# Re-import so edits in the %%writefile cells above take effect.
for name in [m for m in sys.modules if m.startswith("problems.optics.double_slit")]:
    del sys.modules[name]
main = importlib.import_module("problems.optics.double_slit.main")
problem_dir = Path("problems/optics/double_slit")
output = main.run(resolve_params(load_meta(problem_dir), params), Path("out"))
display(Image(filename=str(output)))
